### 📘 学习打卡

##### 请执行下方 cell，点击窗口中的“开始学习”按钮开始本次学习，完成学习后请点击“结束学习”按钮。开始和结束时间都会记录到统一日志文件中，帮助对你的学习情况进行分析。

In [1]:
from projects.clock_in import create_session_widgets
display(create_session_widgets("projects/fa.log"))

# 状态转换图与有限自动机
---

## 概览

在第一个 Notebook 中，我们学习了用正则表达式描述单词模式。但正则表达式只是“描述”，计算机需要一种可执行的模型来“识别”单词——这就是**有限自动机**。

本章从直观的**状态转换图**入手，逐步过渡到形式化的**有限自动机**理论，包括 NFA（非确定有限自动机）和 DFA（确定有限自动机）的定义、表示与模拟算法。你将理解词法分析器在底层是如何工作的——这正是 Lex 等自动生成工具的理论基础。

## 学习目标

1. 理解状态转换图的结构及其在词法分析中的作用
2. 掌握 NFA（非确定有限自动机）的定义、表示和语言接受条件
3. 掌握 DFA（确定有限自动机）的定义、表示和模拟算法
4. 理解 NFA 与 DFA 的核心差异，以及它们在词法分析中的角色

---

## 1. 状态转换图

### 1.1 状态转换图的概念

**状态转换图（Transition Diagram）** 是一种直观地描述词法分析算法的方式。它将单词的识别过程表示为状态之间的迁移。例如，下面就是一个识别关系运算符的状态转换图

<img src="./images/relop_td.png" alt="关系运算符的状态转换图" style="width: 20%; display: block; margin-left: 0;">

**基本要素**：

| 要素 | 含义 |
| :--- | :--- |
| **状态（State）** | 代表已读入（匹配）的符号串所到达的“进度”。如上例中的 0、6、7、8。 |
| **边（Edge）** | 从当前状态读入一个符号后，迁移到另一个状态。如上例中的 0->6、6->7、7->8。 **注意**：7->8 上的 `other` 表示除 7 的其他出射边上符号之外的所有符号|
| **初态（Start State）** | 识别的起点，用箭头指向表示。如上例中的 0。 |
| **终态（Final/Accepting State）** | 识别出一个单词，用双圈表示。如上例中的 7、8。 |

**运转机制**：
1. 从初态开始，逐个读入输入字符
2. 每读入一个字符，根据当前状态和字符决定迁移到哪个状态。例如，对于上例，读入一个符号 `>`，从状态 0 迁移到状态 6
3. 如果符号串中所有符号都处理完毕，最终到达终态，则识别出一个单词。例如，对于上例，继续读入一个符号 `=`，从状态 6 迁移到状态 7，此时符号串处理完毕，表示识别出关系运算符 `>=`
4. 如果无法迁移（无对应的边），则匹配失败，尝试其他模式。例如，对于上例，位于初态时读入一个符号 `1`，找不到对应边，识别失败，表示输入符号串不是关系运算符

**状态对应符号串集合！**：每个状态都存在从初态到它的路径（可能多条），每条路径上的符号连接起来都形成一个符号串，状态即对应所有这些符号串的集合，可理解为**状态即是已匹配某个符号串的进度**。

例如，在上例中：
- 初态：尚未读入任何符号（思考一下，实质上也是对应一个符号串集合`{ε}`，即，初态表示已识别空字的状态）
- 状态 6：对应 `{>}`，已匹配 `>`
- 状态 7：对应 `{>=}`，已识别单词 `>=`
- 状态 8：对应 `{>c | c ≠ =}`，已匹配 `>` 后接任一非 `=` 符号，需要**回退（retract）**最后这个符号（它是下一个单词的开头符号），识别出单词 `<`

### 1.2 状态转换图示例

我们已经了解了状态转换图的基本要素和工作原理，接下来将进一步通过几个例子来展示如何用状态转换图实现更复杂的模式识别过程。

#### 标识符与分隔符的状态转换图

标识符的模式为：**字母开头，后接任意多个字母或数字**。下图是识别标识符的状态转换图。

<img src="./images/id_td.png" alt="标识符与分隔符的状态转换图" style="width: 20%; display: block; margin-left: 0;">

发现与前面例子的区别了吗？本例中要识别的单词是无穷集！

对于“关系运算符”这样的有穷集，每个符号串又都是有穷长度，对应的正则表达式只需连接和并集运算，而状态转换图结构则是从初态到终态的几条简单路径。

但对于无穷集，这样的简单结构就无能为力了，正则表达式需要闭包运算，而状态转换图需要环。

从上图可以看出，状态 9 到状态 10 的边识别“字母开头”，而状态 10 的自环识别“任意多个字母或数字”，状态 10 到状态 11 的边识别出非字母或数字符号（也是需要回退），到达终态，表示标识符识别结束。

**注意**：状态 10 对应的符号串是否是合法的标识符？答案是肯定的。但没有将它设置为为终态。还需继续读入尽可能多的符号，只要还是合法的标识符，直到遇到非字母和数字为止。这就是**最长匹配原则**：尽可能匹配更长的符号串。

下面的状态转换图设计思路类似，它实现了分隔符序列的识别。

<img src="./images/delimit_td.png" alt="标识符与分隔符的状态转换图" style="width: 20%; display: block; margin-left: 0;">

#### 无符号数的状态转换图

无符号数在 Pascal 等语言中的模式为：`digit+ (. digit+)? (E (+|-)? digit+)?`

由于模式较为复杂，用单个状态转换图描述比较困难，可将其分解为**三个子状态转换图**：

**（1）整数**

<img src="./images/int_td.png" alt="无符号数——整数部分的状态转换图" style="width: 27%; display: block; margin-left: 0;">

可以看到，此状态转换图的结构与标识符和分隔符序列的状态转换图是完全一样的，因为它们的模式——无论是自然语言描述还是正规式描述——就是相同结构的。

通过几个例子，你应该对状态转换图与正则表达式的对应关系有所体会：多条路径分岔结构可表示并集运算；一条路径上连续的边可表示连接运算；而环可表示闭包运算。

**注意**：状态转换图，包括接下来要介绍的有限自动机，是可以运转起来实现单词识别的。但它们同样可以充当单词的静态描述，因此可以称为模式，与自然语言描述和正则表达式一样。

**（2）带小数部分**

<img src="./images/frac_td.png" alt="无符号数——小数部分的状态转换图" style="width: 43%; display: block; margin-left: 0;">

可以看到，带小数部分的状态转换图就是两个整数的状态转换图前后连接起来，与正则表达式 `digit+.digit+` 完全对应。

**（3）带指数部分**

<img src="./images/exp_td.png" alt="无符号数——指数部分的状态转换图" style="width: 48%; display: block; margin-left: 0;">

带指数部分的情况更复杂一些，但仍然是路径、分岔、环这几种结构的组合，这是必然的，因为描述单词的正则表达式也只有对应的连接、并集和闭包这几种运算。

可能你觉得边 13->16、16->18 跳过可选的小数部分和指数的符号部分，是前面没见到过的结构。但思考一下，“可选”的概念实际上是并集运算——某个结构有“或”无（`ε`），而这两条边与其他路径形成了分岔，恰好是对应并集运算的。

如何使用这三个状态转换图（主要是应用顺序），正确识别所有可能的无符号数呢？你可能觉得没有头绪，可以考虑一个实际的无符号数 `12.3E4`，思考一下三个状态转换图不同的应用顺序会带来什么样的结果。

容易分析出，正确的匹配顺序是带指数情况 → 带小数情况 → 整数情况。否则的话，上面例子会被错误地识别为 `12` 或 `12.3`，而不是正确的带指数的无符号数。这就是下一节要介绍的**最长匹配原则**。在实际的词法分析器实现中，所有单词的状态转换图可能被合并为一个大的状态转换图，最长匹配原则的实施方式就是“到达终态不停止——转换失败时回退到最后一个终态”。显然，这样即可匹配尽可能长的前缀。

### 1.3 多个状态转换图的使用

在实际的词法分析中，存在多个状态转换图——每个单词类别对应一个，词法分析器如何应用这些状态转换图正确识别单词呢。

**使用原则**：

1. **子集优先原则**：如果单词之间存在集合包含关系，子集通常是优先级更高的特例，应该先应用子集对应的状态转换图。例如，关键字匹配先于标识符匹配，因为关键字集合包含于标识符集合中，前者是特例，后者是一般情况。

2. **最长匹配原则**：当多个模式都能匹配当前输入的前缀时，选择匹配**最长**词素的模式。例如，`<=` 优先于 `<`，`>=` 优先于 `>`，再如上一节无符号整数的例子。

3. **失败后尝试下一个**：当前状态转换图在某个字符处无法继续时，回溯并尝试下一个状态转换图。

**与 Lex 程序的联系**：

Lex 程序的规则段本质上就是一组模式的集合。Lex 的工作方式与状态转换图完全一致：
- Lex 规则的**书写顺序**对应状态转换图的**尝试顺序**
- Lex 的**最长匹配原则**对应状态转换图中“尽可能多读字符再接受”的行为

Lex 程序：
```
<   { ... }
<=  { ... }
>   { ... }
>=  { ... }
```
如果输入为 `<=`，Lex 会优先匹配 `<=` 而不是 `<`，因为 `<=` 更长。如前所述，具体的工作机制是，在状态转换图中先到达 `<` 状态，虽是终态，并不停止；再到达 `<=` 状态，仍不停止；继续读入后续符号，会导致状态迁移失败，此时回退到最后一个终态 `<=`，即为识别出的单词。

---

## 2. 有限自动机

状态转换图直观地描述了单词识别的过程，但它缺乏形式化的数学定义。**有限自动机（Finite Automata）** 正是状态转换图的形式化模型——它为词法分析提供了严格的数学基础，使得我们可以精确地定义“一个词法分析器识别什么语言以及如何识别”，可在此基础上进行算法设计和正确性证明。

有限自动机分为两类：**非确定有限自动机（NFA）** 和 **确定有限自动机（DFA）**。两者能力等价——它们识别的语言类完全相同（正则语言）——但在结构和实现上有显著差异。

### 2.1 NFA：非确定有限自动机

**非确定有限自动机（Nondeterministic Finite Automaton, NFA）** 的“非确定性”体现在：在某个状态下，对于同一个输入符号，可能存在**多个可能的下一状态**，也可能**没有下一状态**；此外，还允许 **ε 边**（不读入任何符号即可迁移）。

#### 2.1.1 NFA 的数学模型

一个 NFA 可以形式化地表示为一个**五元组**：

$M = (S, \Sigma, \delta, s_0, F)$

| 符号 | 含义 |
| :--- | :--- |
| $S$ | 有穷状态集 |
| $\Sigma$ | 有穷字母表（输入符号集合） |
| $\delta$ | 状态转换函数：$S \times (\Sigma \cup \{\epsilon\}) \to 2^S$（映射到状态集的子集） |
| $s_0 \in S$ | 唯一的初态 |
| $F \subseteq S$ | 终态集合（可为空） |

关键区别：$\delta$ 的值域是 $2^S$（S 的所有子集），而不是单个状态。也就是说，从一个状态读入一个符号后，可能有多条边匹配这个符号，可以进入多个状态中的任意一个。它与 ε 边是不确定性的两个来源。

#### 2.1.2 NFA 的表示方式

NFA 有三种等价的表示方式：

**（1）状态转换图**——便于人理解

以识别正规式 `(a|b)*abb` 的 NFA 为例：

<img src="./images/nfa_example.png" alt="(a|b)*abb 的 NFA" style="width: 25%; display: block; margin-left: 0;">

状态 0 有两个自环，匹配 `(a|b)*`。随后则是一条路径 0 → 1 → 2 → 3，匹配 `abb`。可以看到，状态 0 发出了两条标记为 `a` 的边，体现出不确定性。

**（2）状态转换矩阵**——便于计算机存储和处理

行表示当前状态，列表示输入符号，表项为下一状态的集合：

| 状态 | a | b | ε |
| :--- | :--- | :--- | :--- |
| 0 | {0, 1} | {0} | ∅ |
| 1 | ∅ | {2} | ∅ |
| 2 | ∅ | {3} | ∅ |
| 3 | ∅ | ∅ | ∅ |

**（3）转换函数**——严谨的数学描述

$\delta(0, a) = \{0, 1\}, \quad \delta(0, b) = \{0\}, \quad \delta(1, b) = \{2\}, \quad \delta(2, b) = \{3\}$

#### 2.1.3 ε 边的作用

ε 边允许 NFA **在不读入任何符号的情况下**从一个状态迁移到另一个状态。这使得 NFA 可以：
- 表示“可选”结构（如 `a?`）
- 表示“闭包”结构（如 `a*`）
- 将多个子 NFA 组合成更复杂的 NFA

ε 边是 NFA 非确定性的重要来源之一——因为存在 ε 边意味着在当前状态下不读入任何输入，这意味着多个选择——当前状态和 ε 边指向的状态对应相同的符号串，从而造成多条路径共存。

#### 2.1.4 NFA 的工作方式

一个 NFA **接受**一个输入符号串，当且仅当**存在至少一条**从初态到某个终态的路径，其边上标记的符号依次拼接后等于该输入串。

以 `(a|b)*abb` 的 NFA 为例，输入 `aabb`：

```
路径 1：0 →(a) 0 →(a) 1 →(b) 2 →(b) 3  → 接受！
路径 2：0 →(a) 0 →(a) 0 →(b) 0 →(b) 0  → 非终态 0 不意味着不接受！只有所有路径都无法到达终态才有结论“不接受”！
```

由于存在一条到达终态的路径，NFA 接受 `aabb`。

**NFA 的“非确定性”体现在**：在状态 0 读入 `a` 时，可以迁移到 0 或 1。词法分析器需要**同时跟踪所有可能的状态**，而不是只关注一条路径。

NFA M 所定义的语言为：

$L(M) = \{ w \mid \text{存在一条从 } s_0 \text{ 到某个 } f \in F \text{ 的路径，路径上符号依次拼接为 } w \}$

### 2.2 DFA：确定有限自动机

NFA 的“非确定性”虽然使得构造更方便，但直接模拟 NFA 需要同时跟踪一个状态集合，每步的处理开销通常高于只跟踪单个状态。**确定有限自动机（DFA）** 消除了这种非确定性——每个状态对字母表中每个符号有**至多一个**转移，且没有 ε 边。

#### 2.2.1 DFA 的数学模型

一个 DFA 可以形式化地表示为一个**五元组**：

$M = (S, \Sigma, \delta, s_0, F)$

| 符号 | 含义 |
| :--- | :--- |
| $S$ | 有穷状态集 |
| $\Sigma$ | 有穷字母表 |
| $\delta$ | 状态转换函数：$S \times \Sigma \to S$（单值映射） |
| $s_0 \in S$ | 唯一的初态 |
| $F \subseteq S$ | 终态集合 |

与 NFA 的关键区别：
- $\delta$ 的值域是 $S$（单个状态），不是 $2^S$（状态集的子集）
- 没有 ε 边

这意味着 DFA 是**完全确定性**的：在任何状态下读入任何符号，最多只有一个可能的下一状态。

#### 2.2.2 DFA 的表示方式

以识别 `(a|b)*abb` 的 DFA 为例：

<img src="./images/dfa_example.png" alt="(a|b)*abb 的 DFA" style="width: 25%; display: block; margin-left: 0;">

对比 NFA 的版本，DFA 的每个状态对每个符号只有唯一的转移，代价是变得复杂了。

#### 2.2.3 DFA 模拟算法（算法 3.1）

DFA 的实现非常简单——只需要模拟确定的状态转换过程：

```
输入：DFA D，输入串 x（以 eof 结束）
输出：若 D 接受 x 返回 "yes"，否则返回 "no"

s = s0
c = nextchar()
while c != eof:
    s = δ(s, c)
    c = nextchar()
if s ∈ F:
    return "yes"
else:
    return "no"
```

**时间复杂度**：$O(|x|)$，与输入串长度成正比。

**思考一下**：为什么前面没有给出模拟 NFA 运行的算法？

##### 处理未定义状态转换

如果 DFA 的转换函数没有对某些（状态，符号）定义，在实际实现中通常引入一个**死状态（dead state）**——所有未定义的转换都指向它，且死状态的所有转换都指向自身。这样 DFA 的转换函数就成为了全函数。

显然，死状态不会是终态，进入这个状态后会一直原地打转，匹配的符号串都是不被 DFA 接受的，其命名准确地表达了这个特点。

### 2.3 NFA 与 DFA 的对比

| 维度 | NFA | DFA |
| :--- | :--- | :--- |
| **确定性** | 非确定：同一状态同符号可有多条转移，可有 ε 边 | 确定：同一状态同符号至多一条转移，无 ε 边 |
| **状态数** | 通常较少 | 通常较多（可能指数增长） |
| **构造难度** | 容易（从正规式直接构造） | 较难（需要子集构造法转换） |
| **模拟算法** | 需同时跟踪多个状态（状态集），$O(\|x\|*\|S\|^2)$ | 单状态跟踪，$O(\|x\|)$ |
| **内存占用** | 转换函数存储稀疏，占用少 | 转换函数可能很密集，占用多 |

**核心权衡**：NFA 构造方便但运行慢（需跟踪多个状态），DFA 运行快但可能占用更多内存。在实际的词法分析器生成工具（如 Lex）中，通常采用以下策略：

- 用户提供正规式（方便）
- 构造 NFA（构造方便）
- 将 NFA 转换为 DFA（换取运行效率）
- 模拟 DFA 进行词法分析（运行高效）

这也是后续章节的核心内容。

### 课堂思考：为什么不能直接用上面的单状态算法模拟 NFA？

因为非确定有限自动机（Nondeterministic Finite Automaton, NFA）在读入同一个字符后可能到达多个状态，还有不读入字符的 ε 转移。只跟踪一个状态可能漏掉可以接受的路径，所以应当跟踪当前所有可能的状态，并考虑沿 ε 边能到达的状态。输入读完后，只要这些状态中有一个是终态，就接受。

### 2.4 动手练习：自动机设计

本节包含三个自动机设计练习，难度递进。前两个练习要求设计完整的 DFA，第三个练习只需给出 DFA 的几方面要素。

每个练习下方都有 AI 助教窗口，你可以与助教讨论你的设计思路。

#### 练习 2.1：不包含子串 011 的 0/1 串

**题目**：设计一个 DFA，识别所有**不包含子串 `011`** 的 0/1 串。

**设计思路**（自动机运转观点）：从初态开始，随着读入符号，状态不断转移，循此创建新的边和新的状态，直至收敛。

**关键**：创建出的新边，哪些指向新的状态？哪些指向已有状态？

**提示**：回到“状态对应符号串集合”特性，当前状态对应特定符号串集合，读入符号、状态转移，意味着新符号拼接当前符号串末尾，扩展后的符号串集合对应下一状态。符号串集合表示的含义（例如，对于本题，含义就是指“形成011的进度”）是否与某个已有状态一致？是，则符号串集合合并，边指向已有状态；否，创建新状态。状态是否为终态也由符号串集合的含义来决定。

执行下方 cell，向 AI 助教描述你设计的DFA、进行讨论。

##### 我的解答

我用“已经匹配到 `011` 的哪一步”来区分状态。只要已经出现过 `011`，后面再读什么都不能接受。

| 状态 | 含义 | 读入 `0` | 读入 `1` |
| :--- | :--- | :--- | :--- |
| $q_0$ | 没出现 `011`，末尾既不是 `0` 也不是 `01`，包括空串 | $q_1$ | $q_0$ |
| $q_1$ | 没出现 `011`，末尾是 `0` | $q_1$ | $q_2$ |
| $q_2$ | 没出现 `011`，末尾是 `01` | $q_1$ | $q_3$ |
| $q_3$ | 已经出现 `011`，是死状态（dead state） | $q_3$ | $q_3$ |

- 状态集：$S=\{q_0,q_1,q_2,q_3\}$，字母表：$\Sigma=\{0,1\}$。
- 初态是 $q_0$，终态集合是 $F=\{q_0,q_1,q_2\}$，转换关系如表所示。
- 空串还没有出现 `011`，所以初态也是终态。

其中，$q_2$ 读入 `0` 后，末尾变为 `010`，最后的 `0` 又可能成为新的 `011` 的开头，所以回到 $q_1$，不用新建状态。

用两个串检查：

- `0101`：$q_0\to q_1\to q_2\to q_1\to q_2$，最后是终态，接受。
- `0011`：$q_0\to q_1\to q_1\to q_2\to q_3$，最后不是终态，拒绝。


In [2]:
%run projects/problem_ai_helper.py 2.1

D:\Code\fa-homework\projects\problem_ai_helper.py:198: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

##### AI 助教核对

> 以下是本次将上方解答提交给 DeepSeek 助教后得到的真实反馈，保存为文字，重新打开即可阅读。原有助教窗口仍可继续使用。

你的解答正确且完整：状态含义清楚，转换关系正确，包含已出现 `011` 后的死状态 $q_3$，终态集合 $F=\{q_0,q_1,q_2\}$ 也正确。你还特别说明了 $q_2$ 读入 `0` 后回到 $q_1$，抓住了“边可以指向已有状态”的关键点。这个问题你已经掌握得很好，可以结束。

#### 练习 2.2：偶数个 0，偶数个 1 的 0/1 串

**题目**：设计一个 DFA，识别所有**包含偶数个 0 且包含偶数个 1** 的 0/1 串（空串 ε 包含 0 个 0 和 0 个 1，是合法的）。

**设计思路**（状态对应符号串集合观点）：既然状态对应符号串集合，且一个完整的 DFA 的所有状态能匹配字母表上所有符号串。那么可以根据题意，将符号串全集划分为几个子集，分别对应 DFA 的各个状态，然后再构造状态之间的边。

**提示**：本题考察的是 0 和 1 的个数的奇偶性，可据此将全体 0/1 串划分子集、对应 DFA 状态。而边的构造，还是考虑状态迁移导致的符号串变化——0 边和 1 边分别表示在当前符号串末尾拼接了一个 0 或 1，可据此在对应符号串集合（状态）间连接边。

执行下方 cell，向 AI 助教描述你设计的 DFA，进行讨论。

##### 我的解答

只需要记住 `0` 和 `1` 的个数分别是奇数还是偶数，不需要记住具体有多少个。一共有四种组合，因此设置四个状态。

下标中的第一个字母表示 `0` 的个数，第二个表示 `1` 的个数；$E$ 表示偶数，$O$ 表示奇数。

| 状态 | `0` 的个数 | `1` 的个数 | 读入 `0` | 读入 `1` |
| :--- | :--- | :--- | :--- | :--- |
| `q_EE` | 偶数 | 偶数 | `q_OE` | `q_EO` |
| `q_OE` | 奇数 | 偶数 | `q_EE` | `q_OO` |
| `q_EO` | 偶数 | 奇数 | `q_OO` | `q_EE` |
| `q_OO` | 奇数 | 奇数 | `q_EO` | `q_OE` |

- 状态集为 $S=\{q_{EE},q_{OE},q_{EO},q_{OO}\}$，字母表为 $\Sigma=\{0,1\}$。
- 初态是 $q_{EE}$，因为开始时两种字符都是 0 个。
- 唯一的终态是 $q_{EE}$，即 $F=\{q_{EE}\}$。

读到 `0`，只改变 `0` 的奇偶性；读到 `1`，只改变 `1` 的奇偶性，所以转换关系就是表中的结果。

简单检查：空串停在 $q_{EE}$，接受；`0101` 的状态变化为 $q_{EE}\to q_{OE}\to q_{OO}\to q_{EO}\to q_{EE}$，接受；`001` 最后停在 $q_{EO}$，有奇数个 `1`，所以拒绝。


In [3]:
%run projects/problem_ai_helper.py 2.2

D:\Code\fa-homework\projects\problem_ai_helper.py:198: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

##### AI 助教核对

> 以下是本次将上方解答提交给 DeepSeek 助教后得到的真实反馈，保存为文字，重新打开即可阅读。原有助教窗口仍可继续使用。

你的解答完全正确。

你用四个状态表示 `0` 的个数和 `1` 的个数的奇偶性组合，状态转移也准确：读入 `0` 只翻转“0 的奇偶性”，读入 `1` 只翻转“1 的奇偶性”。初态和唯一终态都取 $q_{EE}$，因此空串被接受，且最终正好识别“偶数个 0 且偶数个 1”的串。例子检查也说明你理解了自动机的运行过程。

#### 练习 2.3：狼、羊、白菜

**题目**：一个农夫要带狼、羊、白菜过河。船只能容纳农夫和一样东西（农夫不在船上船不能动）。狼会吃羊，羊会吃白菜，如果农夫不在场，狼和羊不能单独在一起，羊和白菜不能单独在一起。请用有限自动机的思想来描述这个问题的求解过程。

**要求**：本题只是讨论用 DFA 求解此问题的思路，不要求画出完整的 DFA 以及给出最终的解。请思考并回答以下问题：

1. **状态应该表示什么，如果表达**？完整 DFA 应该有几个状态？（提示：农夫、狼、羊、白菜的当前位置（在哪个岸边）就描述了当前状态。四者的位置共有多少种组合？其中多少种是安全、合法的状态？）
2. **状态间的边表示什么**？（提示：农夫、狼、羊、白菜从当前位置到下一个位置，这种变化是如何产生的）
3. **初态和终态分别是什么**？（提示：根据第1问，自然是初始时的位置和完成渡河之后的位置）
4. **一个渡河方案对应 DFA 中的什么**？（提示：渡河方案就是从初始位置变成渡河之后位置的方法，而位置对应 DFA 状态）

执行下方 cell，进入 AI 助教窗口讨论你的理解。

##### 我的解答

**1. 状态表示四个对象分别在哪一岸。**

用 $(f,w,s,c)$ 表示农夫、狼、羊、白菜的位置，`0` 表示左岸，`1` 表示右岸。例如 $(0,0,0,0)$ 表示都在左岸。

四个对象各有两个位置，一共 $2^4=16$ 种组合，但不是每种组合都安全：

- 农夫和羊在同一岸时，狼、白菜各在哪一岸都不会出事，共 $2\times2\times2=8$ 种。
- 农夫和羊不在同一岸时，狼、白菜都必须和农夫在同一岸，否则会吃羊或吃白菜，共 2 种。

所以用 **10 个安全状态**表示合法局面。如果要求每个状态对每种输入都有转换，可以另加一个拒绝状态，把非法动作都转到它，这种完整表示共有 11 个状态；16 是原始位置组合数，不是安全状态数。

**2. 边表示一次渡河动作。**

可以把输入符号规定为“农夫单独过河”“带狼”“带羊”“带白菜”。每次农夫一定换岸，同行的对象也换岸，其余对象不动。带的对象必须原来就和农夫在同一岸，而且渡河后的局面必须安全，否则不是合法的边。

给定当前状态和所选动作，下一状态是唯一的，因此可以按确定有限自动机（Deterministic Finite Automaton, DFA）来表示。

**3. 初态和终态。**

初态为 $(0,0,0,0)$，终态为 $(1,1,1,1)$。前者表示都没有过河，后者表示都已到达对岸。

**4. 渡河方案对应什么。**

一个可行的方案对应从初态到终态的一条路径，路径上的边依次表示每一步怎么渡河。沿途只能经过安全状态。本题只讨论这种表示方法，不展开完整状态图和具体渡河步骤。


In [4]:
%run projects/problem_ai_helper.py 2.3

D:\Code\fa-homework\projects\problem_ai_helper.py:198: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.input_box.on_submit(self.send_message)


<IPython.core.display.Javascript object>

##### AI 助教核对

> 以下是本次将上方解答提交给 DeepSeek 助教后得到的真实反馈，保存为文字，重新打开即可阅读。原有助教窗口仍可继续使用。

你的回答已经抓住了核心：  

1. **状态表示四个对象各自在哪一岸**，并进一步区分了“16 种位置组合”和“合法安全状态”，这个理解很准确。  
2. **边表示一次渡河动作**，也就是导致农夫和可能带着的东西换岸、从而让状态发生变化的操作，这也正确。  
3. **初态是 $(0,0,0,0)$，终态是 $(1,1,1,1)$**，对应出发前和全部过河后，正确。  
4. **渡河方案对应从初态到终态的一条路径**，路径上的边就是每一步渡河动作，这也正是 DFA 中状态变化的过程。  

你还提到可以加一个拒绝状态来处理非法动作，这是可选但合理的完整化建模方式。整体上四个问题都覆盖到了，思路清晰。